In [25]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine,text

load_dotenv()

True

In [ ]:
import http.client
conn = http.client.HTTPSConnection("api.collectapi.com")

headers = {
    'content-type': "application/json",
    'authorization': os.getenv('API_KEY')
    }

conn.request("GET", "/gasPrice/stateUsaPrice?state=WA", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

'{"success":true,"result":{"state":[{"currency":"usd","name":"Washington","lowername":"washington","gasoline":"5.5670","midGrade":"5.8530","premium":"6.1020","diesel":"6.5330"}],"cities":[{"currency":"usd","gasoline":"5.4760","midGrade":"5.7350","premium":"5.9010","diesel":"6.2200","name":"Bellingham","lowername":"bellingham"},{"currency":"usd","gasoline":"5.7290","midGrade":"6.0120","premium":"6.2320","diesel":"6.5530","name":"Bremerton","lowername":"bremerton"},{"currency":"usd","gasoline":"5.5370","midGrade":"5.7600","premium":"6.0650","diesel":"6.7290","name":"Olympia","lowername":"olympia"},{"currency":"usd","gasoline":"5.1990","midGrade":"5.4900","premium":"5.7680","diesel":"6.2060","name":"Richland-Kennewick-Pasco","lowername":"richland-kennewick-pasco"},{"currency":"usd","gasoline":"5.8330","midGrade":"6.0830","premium":"6.3260","diesel":"6.8040","name":"Seattle-Bellevue-Everett","lowername":"seattle-bellevue-everett"},{"currency":"usd","gasoline":"5.1100","midGrade":"5.3980","

In [ ]:
type(data)

str

In [ ]:
# str to dict
data = json.loads(data)
type(data)

dict

In [13]:
data.keys()

dict_keys(['success', 'result'])

In [17]:
type(data['result']['cities'])

list

In [ ]:
# extract city level data
gas_data = data['result']['cities']
gas_data[:3]

[{'currency': 'usd',
  'gasoline': '5.4760',
  'midGrade': '5.7350',
  'premium': '5.9010',
  'diesel': '6.2200',
  'name': 'Bellingham',
  'lowername': 'bellingham'},
 {'currency': 'usd',
  'gasoline': '5.7290',
  'midGrade': '6.0120',
  'premium': '6.2320',
  'diesel': '6.5530',
  'name': 'Bremerton',
  'lowername': 'bremerton'},
 {'currency': 'usd',
  'gasoline': '5.5370',
  'midGrade': '5.7600',
  'premium': '6.0650',
  'diesel': '6.7290',
  'name': 'Olympia',
  'lowername': 'olympia'}]

In [20]:
#extract city level data, drop the lowername column, rename name to cities  then you store to db
gas_df = pd.DataFrame(gas_data)
gas_df.head()

,currency,gasoline,midGrade,premium,diesel,name,lowername
0,usd,5.4760,5.7350,5.9010,6.2200,Bellingham,bellingham
1,usd,5.7290,6.0120,6.2320,6.5530,Bremerton,bremerton
2,usd,5.5370,5.7600,6.0650,6.7290,Olympia,olympia
3,usd,5.1990,5.4900,5.7680,6.2060,Richland-Kennewick-Pasco,richland-kennewick-pasco
4,usd,5.8330,6.0830,6.3260,6.8040,Seattle-Bellevue-Everett,seattle-bellevue-everett


In [22]:
# drop lowername
gas_df=gas_df.drop(columns=["lowername"])
gas_df.head()

,currency,gasoline,midGrade,premium,diesel,name
0,usd,5.4760,5.7350,5.9010,6.2200,Bellingham
1,usd,5.7290,6.0120,6.2320,6.5530,Bremerton
2,usd,5.5370,5.7600,6.0650,6.7290,Olympia
3,usd,5.1990,5.4900,5.7680,6.2060,Richland-Kennewick-Pasco
4,usd,5.8330,6.0830,6.3260,6.8040,Seattle-Bellevue-Everett


In [23]:
# name to cities
gas_df.rename(columns={"name": "cities"}, inplace=True)
gas_df.head()

,currency,gasoline,midGrade,premium,diesel,cities
0,usd,5.4760,5.7350,5.9010,6.2200,Bellingham
1,usd,5.7290,6.0120,6.2320,6.5530,Bremerton
2,usd,5.5370,5.7600,6.0650,6.7290,Olympia
3,usd,5.1990,5.4900,5.7680,6.2060,Richland-Kennewick-Pasco
4,usd,5.8330,6.0830,6.3260,6.8040,Seattle-Bellevue-Everett


In [26]:
# store in db
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

with engine.connect() as conn:
        result = conn.execute(text('select 1;'))
        for i in result:
            print(i)

gas_df.to_sql('gas_prices',  engine, schema='public', if_exists='replace', index=False)

(1,)


14